# Text Generation using RNN, LSTM and GRU
This notebook trains a Simple RNN, an LSTM and a GRU on the same text data and compares their next word prediction quality.

## Problem Statement
Build a next word prediction model using RNN, LSTM and GRU and compare their training loss and generated text.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense
import numpy as np
import matplotlib.pyplot as plt

print("tensorflow version:", tf.__version__)

## Load Text Corpus
A slightly longer paragraph is used so the models have enough context to learn from.

In [ ]:
corpus = """
deep learning is a subset of machine learning that uses neural networks with many layers
recurrent neural networks are useful for sequential data like text and time series
a simple rnn struggles to remember information from many steps in the past
this problem is called the vanishing gradient problem
lstm networks were designed to solve this problem using gates
the forget gate decides what information to remove from the cell state
the input gate decides what new information to add to the cell state
the output gate decides what part of the cell state to output
gru networks are a simpler alternative to lstm networks
a gru uses only two gates called the update gate and the reset gate
gru networks train faster than lstm networks and often give similar results
text generation models learn to predict the next word given previous words
a good text generation model can produce sentences that sound natural
training on more text usually leads to better and more coherent generated text
"""

print(corpus)

## Tokenization and Sequence Creation
The text is converted into integer tokens, then split into input and target word sequences for next word prediction.

In [ ]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts([corpus])

total_words = len(tokenizer.word_index) + 1
print("vocabulary size:", total_words)

input_sequences = []
for line in corpus.split("\n"):
    if line.strip() == "":
        continue
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_seq = token_list[:i+1]
        input_sequences.append(n_gram_seq)

max_len = max(len(seq) for seq in input_sequences)
input_sequences = pad_sequences(input_sequences, maxlen=max_len, padding="pre")

X = input_sequences[:, :-1]
y = input_sequences[:, -1]

print("X shape:", X.shape)
print("y shape:", y.shape)

## Train Validation Split
A small validation split is kept aside to check if the models are actually learning and not just memorizing.

In [ ]:
num_samples = X.shape[0]
split_index = int(num_samples * 0.85)

# shuffle before splitting so the validation set is not just the last few lines
indices = np.arange(num_samples)
np.random.seed(42)
np.random.shuffle(indices)

X = X[indices]
y = y[indices]

X_train, X_val = X[:split_index], X[split_index:]
y_train, y_val = y[:split_index], y[split_index:]

print("train samples:", X_train.shape[0])
print("val samples:", X_val.shape[0])

## Model 1: Vanilla RNN
This is the baseline model. It usually struggles with long term dependencies.

In [ ]:
rnn_model = Sequential([
    Embedding(total_words, 32),
    SimpleRNN(64),
    Dense(total_words, activation="softmax")
])

rnn_model.compile(loss="sparse_categorical_crossentropy",
                   optimizer="adam",
                   metrics=["accuracy"])

rnn_history = rnn_model.fit(X_train, y_train,
                             validation_data=(X_val, y_val),
                             epochs=150, verbose=0)

print("rnn training completed")
print("rnn param count:", rnn_model.count_params())

## Model 2: LSTM
LSTM uses input, forget and output gates to keep long term memory.

In [ ]:
lstm_model = Sequential([
    Embedding(total_words, 32),
    LSTM(64),
    Dense(total_words, activation="softmax")
])

lstm_model.compile(loss="sparse_categorical_crossentropy",
                    optimizer="adam",
                    metrics=["accuracy"])

lstm_history = lstm_model.fit(X_train, y_train,
                               validation_data=(X_val, y_val),
                               epochs=150, verbose=0)

print("lstm training completed")
print("lstm param count:", lstm_model.count_params())

## Model 3: GRU
GRU uses only an update gate and a reset gate, so it trains faster than LSTM.

In [ ]:
gru_model = Sequential([
    Embedding(total_words, 32),
    GRU(64),
    Dense(total_words, activation="softmax")
])

gru_model.compile(loss="sparse_categorical_crossentropy",
                   optimizer="adam",
                   metrics=["accuracy"])

gru_history = gru_model.fit(X_train, y_train,
                             validation_data=(X_val, y_val),
                             epochs=150, verbose=0)

print("gru training completed")
print("gru param count:", gru_model.count_params())

## Compare Training and Validation Loss
Plotting the loss curves shows which model fits the data best and which one overfits the most.

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(rnn_history.history["loss"], label="rnn train")
plt.plot(lstm_history.history["loss"], label="lstm train")
plt.plot(gru_history.history["loss"], label="gru train")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("training loss comparison")
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(rnn_history.history["val_loss"], label="rnn val")
plt.plot(lstm_history.history["val_loss"], label="lstm val")
plt.plot(gru_history.history["val_loss"], label="gru val")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.title("validation loss comparison")
plt.legend()
plt.show()

## Text Generation Function
This function predicts the next word again and again to build a full sentence. Temperature controls how random the word choice is.

In [ ]:
def sample_word(probs, temperature=1.0):
    probs = np.asarray(probs).astype("float64")
    probs = np.log(probs + 1e-8) / temperature
    exp_probs = np.exp(probs)
    probs = exp_probs / np.sum(exp_probs)
    return np.random.choice(len(probs), p=probs)

def generate_text(model, seed_text, next_words=8, temperature=0.8):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len-1, padding="pre")
        predicted_probs = model.predict(token_list, verbose=0)[0]
        predicted_index = sample_word(predicted_probs, temperature)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted_index:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text

## Generate Text Samples
Same seed text is passed to all three models to compare the quality of generated sentences.

In [ ]:
seed = "deep learning is"

print("rnn :", generate_text(rnn_model, seed))
print("lstm:", generate_text(lstm_model, seed))
print("gru :", generate_text(gru_model, seed))

## Student Learning Tasks
1. Replace the corpus with a longer paragraph of your own
2. Increase the embedding dimension from 32 to 64
3. Change hidden units from 64 to 128 and check if loss improves
4. Try different temperature values in generate_text
5. Compare param count vs training time for all three models

## Conclusion
- vanilla rnn trains fastest but has the highest validation loss because it struggles with longer context
- lstm has the most parameters and generally gets the lowest loss on this corpus
- gru gets close to lstm performance with fewer parameters, so it trains a bit faster
- validation loss is what actually shows generalization, training loss alone can be misleading